In [9]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)
# 1.⁠ ⁠State Definition
# The assistant maintains a notebook-like state for each learner:
# •⁠  ⁠topic → The subject the learner is studying (e.g., "Photosynthesis").
# •⁠  ⁠flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
# •⁠  ⁠progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

# 2.⁠ ⁠Workflow (Graph of States)
# Each learner interaction flows through nodes until a flashcard is delivered:
# •⁠  ⁠Input Node
# •⁠  ⁠Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
# •⁠  ⁠State updates: topic = "cell biology"
# •⁠  ⁠Generation Node (Groq API)
# •⁠  ⁠Groq generates a flashcard:
# •⁠  ⁠flashcard.question = "What organelle is known as the powerhouse of the cell?"
# •⁠  ⁠flashcard.answer = "Mitochondria"
# •⁠  ⁠Response Node
# •⁠  ⁠Assistant presents the flashcard question to the learner.
# •⁠  ⁠Evaluation Node
# •⁠  ⁠Learner responds with their answer.
# •⁠  ⁠Assistant checks correctness and updates progress.
# •⁠  ⁠History Node
# •⁠  ⁠Logs the flashcard attempt:
# •⁠  ⁠progress = [{question, learner_answer, correct/incorrect}]
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests

# 1. Define State
class StudyState(TypedDict):
    topic: str
    flashcard: dict
    learner_answer: str
    progress: list


# 2. Helper Function → Groq API Call
def generate_flashcard_from_groq(topic):
    api_key = "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"

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": f"Generate one flashcard question and answer on topic: {topic}. "
                               f"Return in format Question: ... Answer: ..."
                }
            ]
        }
    )

    response_json = response.json()

    if "choices" not in response_json:
        print("Error: 'choices' key not found in Groq API response.")
        print("Full API Response:", response_json)
        raise KeyError("'choices' key not found in Groq API response")

    result = response_json["choices"][0]["message"]["content"]

    lines = result.split("Answer:")
    question = lines[0].replace("Question:", "").strip()
    answer = lines[1].strip()

    return {
        "question": question,
        "answer": answer
    }


# 3. Generation Node
def generate_flashcard(state: StudyState):
    card = generate_flashcard_from_groq(state["topic"])

    return {
        "flashcard": card
    }


# 4. Response Node
def present_flashcard(state: StudyState):
    print("\nFlashcard Question:")
    print(state["flashcard"]["question"])

    user_ans = input("Your Answer: ")

    return {
        "learner_answer": user_ans
    }


# 5. Evaluation Node
def evaluate_answer(state: StudyState):
    correct_answer = state["flashcard"]["answer"]
    learner_answer = state["learner_answer"]

    is_correct = learner_answer.lower() == correct_answer.lower()

    attempt = {
        "question": state["flashcard"]["question"],
        "learner_answer": learner_answer,
        "correct_answer": correct_answer,
        "result": "Correct" if is_correct else "Incorrect"
    }

    return {
        "progress": state["progress"] + [attempt]
    }


# 6. History Node
def show_progress(state: StudyState):
    print("\nProgress Log:")
    for item in state["progress"]:
        print(item)

    return state


# 7. Build Graph
graph = StateGraph(StudyState)

graph.add_node("generate_flashcard", generate_flashcard)
graph.add_node("present_flashcard", present_flashcard)
graph.add_node("evaluate_answer", evaluate_answer)
graph.add_node("show_progress", show_progress)

graph.set_entry_point("generate_flashcard")

graph.add_edge("generate_flashcard", "present_flashcard")
graph.add_edge("present_flashcard", "evaluate_answer")
graph.add_edge("evaluate_answer", "show_progress")
graph.add_edge("show_progress", END)

app = graph.compile()


# 8. Run Example
initial_state = {
    "topic": "cell biology",
    "flashcard": {},
    "learner_answer": "",
    "progress": []
}

result = app.invoke(initial_state)


Flashcard Question:
What is the primary function of the mitochondria in a cell?
Your Answer: power house of cell

Progress Log:
{'question': 'What is the primary function of the mitochondria in a cell?', 'learner_answer': 'power house of cell', 'correct_answer': 'The primary function of the mitochondria is to generate energy for the cell through the process of cellular respiration.', 'result': 'Incorrect'}


In [11]:
# Scenario: AI-Powered Project Tracker (Task-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each project:
# - task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
# - status → The current state of the task (e.g., "in progress", "completed", "blocked").
# - log → A history of all updates, including who made them and when.

# 2. Workflow (Graph of States)
# Each project update flows through nodes until the task status is refreshed:
# - Input Node
# - Team member submits an update (e.g., "Report draft completed").
# - State updates: task = "Q1 financial report"
# - Processing Node (Groq API)
# - Groq interprets the update and assigns a status:
# - status = "completed"
# - Response Node
# - Assistant confirms the update back to the team:
# - "Task Q1 financial report marked as completed."
# - History Node
# - Logs the update:
# - log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from datetime import datetime


# 1. State Definition
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    log: list


# 2. Groq API Helper Function
def get_status_from_groq(task, update):
    api_key = "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"

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": f"""
                    Task: {task}
                    Update: {update}

                    Determine task status from this update.
                    Possible outputs:
                    - in progress
                    - completed
                    - blocked

                    Return only one status word.
                    """
                }
            ]
        }
    )

    response_json = response.json()

    if "choices" not in response_json:
        print("Error: 'choices' key not found in Groq API response.")
        print("Full API Response:", response_json)
        raise KeyError("'choices' key not found in Groq API response")

    status = response_json["choices"][0]["message"]["content"].strip().lower()

    return status


# 3. Input Node
def input_update(state: ProjectState):
    task_name = input("Enter task name: ")
    update_msg = input("Enter update: ")

    return {
        "task": task_name,
        "update": update_msg
    }


# 4. Processing Node
def process_update(state: ProjectState):
    status = get_status_from_groq(state["task"], state["update"])

    return {
        "status": status
    }


# 5. Response Node
def confirm_update(state: ProjectState):
    print(f"\nTask '{state['task']}' marked as {state['status']}.")

    return state


# 6. History Node
def update_log(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "log": state["log"] + [entry]
    }


# 7. Build Graph
graph = StateGraph(ProjectState)

graph.add_node("input_update", input_update)
graph.add_node("process_update", process_update)
graph.add_node("confirm_update", confirm_update)
graph.add_node("update_log", update_log)

graph.set_entry_point("input_update")

graph.add_edge("input_update", "process_update")
graph.add_edge("process_update", "confirm_update")
graph.add_edge("confirm_update", "update_log")
graph.add_edge("update_log", END)

app = graph.compile()


# 8. Run Graph
initial_state = {
    "task": "",
    "update": "",
    "status": "",
    "log": []
}

result = app.invoke(initial_state)

print("\nUpdated Log:")
print(result["log"])

Enter task name: Q1 financial report
Enter update: Report draft completed

Task 'Q1 financial report' marked as completed.

Updated Log:
[{'task': 'Q1 financial report', 'update': 'Report draft completed', 'status': 'completed', 'timestamp': '2026-04-02 06:13:36'}]


In [ ]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests


# 1. State Definition
class SymptomState(TypedDict):
    symptom: str
    observations: list
    analysis: str
    recommendation: str
    steps_done: int


# 2. Groq API Helper
def get_observation_from_groq(symptom):
    api_key = "YOUR_GROQ_API_KEY"

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}"
        },
        json={
            "model": "llama3-8b-8192",
            "messages": [
                {
                    "role": "user",
                    "content": f"""
                    Symptom: {symptom}
                    Give one general possible observation or related condition.
                    Keep it short.
                    """
                }
            ]
        }
    )

    return response.json()["choices"][0]["message"]["content"].strip()


# 3. Symptom Input Node
def input_symptom(state: SymptomState):
    symptom = input("Enter symptom: ")

    return {
        "symptom": symptom
    }


# 4. Observation Node
def generate_observation(state: SymptomState):
    note = get_observation_from_groq(state["symptom"])

    return {
        "observations": state["observations"] + [note],
        "steps_done": state["steps_done"] + 1
    }


# 5. Analysis Node
def analyze_observations(state: SymptomState):
    summary = " | ".join(state["observations"])

    return {
        "analysis": f"Possible related factors: {summary}"
    }


# 6. Conditional Check
def check_observations(state: SymptomState):
    if len(state["observations"]) < 3:
        return "more_observations"
    return "enough_observations"


# 7. Recommendation Node
def give_recommendation(state: SymptomState):
    recommendation = (
        f"For symptom '{state['symptom']}', "
        f"please consult a doctor if symptoms persist or worsen."
    )

    return {
        "recommendation": recommendation
    }


# 8. End Node
def final_response(state: SymptomState):
    print("\nFinal Recommendation:")
    print(state["recommendation"])

    return state


# 9. Build Graph
graph = StateGraph(SymptomState)

graph.add_node("input_symptom", input_symptom)
graph.add_node("generate_observation", generate_observation)
graph.add_node("analyze_observations", analyze_observations)
graph.add_node("give_recommendation", give_recommendation)
graph.add_node("final_response", final_response)

graph.set_entry_point("input_symptom")

graph.add_edge("input_symptom", "generate_observation")
graph.add_edge("generate_observation", "analyze_observations")

graph.add_conditional_edges(
    "analyze_observations",
    check_observations,
    {
        "more_observations": "generate_observation",
        "enough_observations": "give_recommendation"
    }
)

graph.add_edge("give_recommendation", "final_response")
graph.add_edge("final_response", END)

app = graph.compile()


# 10. Run Graph
initial_state = {
    "symptom": "",
    "observations": [],
    "analysis": "",
    "recommendation": "",
    "steps_done": 0
}

result = app.invoke(initial_state)

In [13]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests


# 1. State Definition
class SymptomState(TypedDict):
    symptom: str
    observations: list
    analysis: str
    recommendation: str
    steps_done: int


# 2. Groq API Helper
def get_observation_from_groq(symptom):
    api_key = "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"

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": f"""
                    Symptom: {symptom}
                    Give one general possible observation or related condition.
                    Keep it short.
                    """
                }
            ]
        }
    )

    response_json = response.json()

    if "choices" not in response_json:
        print("Error: 'choices' key not found in Groq API response.")
        print("Full API Response:", response_json)
        raise KeyError("'choices' key not found in Groq API response")

    return response_json["choices"][0]["message"]["content"].strip()


# 3. Symptom Input Node
def input_symptom(state: SymptomState):
    symptom = input("Enter symptom: ")

    return {
        "symptom": symptom
    }


# 4. Observation Node
def generate_observation(state: SymptomState):
    note = get_observation_from_groq(state["symptom"])

    return {
        "observations": state["observations"] + [note],
        "steps_done": state["steps_done"] + 1
    }


# 5. Analysis Node
def analyze_observations(state: SymptomState):
    summary = " | ".join(state["observations"])

    return {
        "analysis": f"Possible related factors: {summary}"
    }


# 6. Conditional Check
def check_observations(state: SymptomState):
    if len(state["observations"]) < 3:
        return "more_observations"
    return "enough_observations"


# 7. Recommendation Node
def give_recommendation(state: SymptomState):
    recommendation = (
        f"For symptom '{state['symptom']}', "
        f"please consult a doctor if symptoms persist or worsen."
    )

    return {
        "recommendation": recommendation
    }


# 8. End Node
def final_response(state: SymptomState):
    print("\nFinal Recommendation:")
    print(state["recommendation"])

    return state


# 9. Build Graph
graph = StateGraph(SymptomState)

graph.add_node("input_symptom", input_symptom)
graph.add_node("generate_observation", generate_observation)
graph.add_node("analyze_observations", analyze_observations)
graph.add_node("give_recommendation", give_recommendation)
graph.add_node("final_response", final_response)

graph.set_entry_point("input_symptom")

graph.add_edge("input_symptom", "generate_observation")
graph.add_edge("generate_observation", "analyze_observations")

graph.add_conditional_edges(
    "analyze_observations",
    check_observations,
    {
        "more_observations": "generate_observation",
        "enough_observations": "give_recommendation"
    }
)

graph.add_edge("give_recommendation", "final_response")
graph.add_edge("final_response", END)

app = graph.compile()


# 10. Run Graph
initial_state = {
    "symptom": "",
    "observations": [],
    "analysis": "",
    "recommendation": "",
    "steps_done": 0
}

result = app.invoke(initial_state)

Enter symptom: persistent cough

Final Recommendation:
For symptom 'persistent cough', please consult a doctor if symptoms persist or worsen.


In [14]:
# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('groq')
    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def draft_email(state: EmailState):
    print(f"📝 Drafting email for task: {state['task']}")
    draft = groq_call(f"Draft a professional email regarding: {state['task']}")
    return {"draft": draft}

def human_review(state: EmailState):

    print(f"📧 Draft ready for review:\n\n{state['draft']}\n")
    return {}

def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}

# 4. Build Graph
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Draft email
app.invoke({"task": "Q3 Report", "draft": "", "approved": False}, thread)

# Step 2: Human reviews draft and resumes
app.invoke({"approved": True}, thread)

📝 Drafting email for task: Q3 Report
📝 Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': 'Subject: Q3 Report and Performance Update\n\nDear Team,\n\nAs we approach the end of Q3, I would like to take a moment to review our current progress and discuss the upcoming challenges. As you know, our Q3 performance will play a crucial role in shaping our overall business strategy for the remainder of the year.\n\nBelow are the key highlights from our Q3 report:\n\n- [Revenue/ Sales Performance]: We have seen a growth of [X]% compared to the same period last year, with total revenue reaching [amount]. This increase is largely attributed to our successful product launches and expansion into new markets.\n- [Key Milestones]: Our core product, [product name], has reached [milestone], demonstrating significant traction in the industry.\n- [Expenses and Efficiency]: Our operating expenses have been managed effectively, with a reduction of [X]% compared to Q3 of last year. This is a result of our continuous efforts to optimize processes and streamline oper

In [17]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from datetime import datetime
import requests


# 1. State Definition
class ReportState(TypedDict):
    topic: str
    draft_report: str
    review_status: str
    reviewer_comments: str
    final_status: str
    audit_log: list


# 2. Groq Helper Function
def generate_report_with_groq(topic):
    api_key = "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"

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": f"Generate a short structured report on {topic}"
                }
            ]
        }
    )

    response_json = response.json()

    if "choices" not in response_json:
        print("Error: 'choices' key not found in Groq API response.")
        print("Full API Response:", response_json)
        raise KeyError("'choices' key not found in Groq API response")

    return response_json["choices"][0]["message"]["content"]


# 3. Input Node
def input_topic(state: ReportState):
    topic = input("Enter report topic: ")

    return {
        "topic": topic
    }


# 4. Draft Generation Node
def draft_report(state: ReportState):
    report = generate_report_with_groq(state["topic"])

    print("\nGenerated Draft Report:\n")
    print(report)

    return {
        "draft_report": report
    }


# 5. Human Review Node
def human_review(state: ReportState):
    print("\n--- Human Review Required ---")

    decision = input("Approve or Reject? ").lower()
    comments = input("Enter reviewer comments: ")

    return {
        "review_status": decision,
        "reviewer_comments": comments
    }


# 6. Conditional Decision
def review_decision(state: ReportState):
    if state["review_status"] == "approve":
        return "approved"
    return "rejected"


# 7. Publish Node
def publish_report(state: ReportState):
    print("\nReport published successfully.")

    return {
        "final_status": "published"
    }


# 8. Reject Node
def reject_report(state: ReportState):
    print("\nReport rejected. Needs revision.")

    return {
        "final_status": "rejected"
    }


# 9. Audit Log Node
def log_action(state: ReportState):
    entry = {
        "topic": state["topic"],
        "status": state["final_status"],
        "comments": state["reviewer_comments"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "audit_log": state["audit_log"] + [entry]
    }


# 10. Build Workflow Graph
graph = StateGraph(ReportState)

graph.add_node("input_topic", input_topic)
graph.add_node("draft_report", draft_report)
graph.add_node("human_review", human_review)
graph.add_node("publish_report", publish_report)
graph.add_node("reject_report", reject_report)
graph.add_node("log_action", log_action)

graph.set_entry_point("input_topic")

graph.add_edge("input_topic", "draft_report")
graph.add_edge("draft_report", "human_review")

graph.add_conditional_edges(
    "human_review",
    review_decision,
    {
        "approved": "publish_report",
        "rejected": "reject_report"
    }
)

graph.add_edge("publish_report", "log_action")
graph.add_edge("reject_report", "log_action")
graph.add_edge("log_action", END)

app = graph.compile()


# 11. Initial State
initial_state = {
    "topic": "",
    "draft_report": "",
    "review_status": "",
    "reviewer_comments": "",
    "final_status": "",
    "audit_log": []
}


# 12. Run
result = app.invoke(initial_state)

print("\nAudit Log:")
print(result["audit_log"])

Enter report topic: Q1 sales report

Generated Draft Report:

**Q1 Sales Report**

**Date:** January 1, 2024 - March 31, 2024

**Introduction:**
Our Q1 sales report provides an overview of the company's sales performance for the first quarter of 2024. The report highlights key sales trends, achievements, and areas for improvement.

**Summary of Key Sales Metrics:**

1. **Total Sales:** $12,567,000 (up 8% from Q1 2023)
2. **Revenue Growth:** 8% (above the target of 5%)
3. **Gross Margin:** 40% (in line with Q1 2023)
4. **Average Sales Order Value:** $2,500 (up 12% from Q1 2023)
5. **Customer Acquisition:** 20 new customers added during Q1

**Sales Performance by Region:**

1. **North America:** $6,123,000 (up 10% from Q1 2023)
2. **Europe:** $3,542,000 (up 5% from Q1 2023)
3. **Asia-Pacific:** $1,902,000 (up 15% from Q1 2023)
4. **Latin America:** $900,000 (up 10% from Q1 2023)

**Key Achievements:**

1. **Strong Demand for New Product Launches:** Our new product launches saw significan